# ConvLSTM Training — Kolkata NTL (GPU)
Self-contained notebook. Attach the `ntl-kolkata-convlstm` dataset before running.
Enable **GPU T4 x2** in Settings → Accelerator.

In [ ]:
# ── 0. Verify GPU ──────────────────────────────────────────────────────────
import tensorflow as tf
tf.config.threading.set_intra_op_parallelism_threads(4)
tf.config.threading.set_inter_op_parallelism_threads(2)
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs available: {gpus}')
print(f'TensorFlow version: {tf.__version__}')

In [ ]:
# ── 1. Imports ─────────────────────────────────────────────────────────────
import os, json
import numpy as np
import joblib
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers

# ── Paths (dataset attached as 'ntl-kolkata-convlstm') ────────────────────
DATASET_SLUG = 'ntl-kolkata-convlstm'   # <-- match exactly what you named it on Kaggle
INPUT_DIR    = f'/kaggle/input/{DATASET_SLUG}'
OUTPUT_DIR   = '/kaggle/working'

FRAMES_NPZ = os.path.join(INPUT_DIR, 'frames.npz')
SCALER_PKL = os.path.join(INPUT_DIR, 'frame_scaler.pkl')
META_PATH  = os.path.join(INPUT_DIR, 'frame_metadata.json')
MODEL_OUT  = os.path.join(OUTPUT_DIR, 'convlstm_model.keras')
HIST_OUT   = os.path.join(OUTPUT_DIR, 'training_history.json')

# ── Hyperparameters ────────────────────────────────────────────────────────
FILTERS_ENC = (8, 16, 16)
FILTERS_DEC = (16, 8)
KERNEL_SIZE = 3
DROPOUT     = 0.0
LR          = 5e-4
EPOCHS      = 500
BATCH_SIZE  = 16
VAL_SPLIT   = 12
PATIENCE    = 20
SEED        = 42

tf.random.set_seed(SEED)
np.random.seed(SEED)
print('Paths and hyperparameters set.')

In [ ]:
# ── 2. Custom metric ───────────────────────────────────────────────────────
@tf.keras.utils.register_keras_serializable(package='convlstm')
def pixel_rmse(y_true, y_pred):
    return tf.sqrt(tf.reduce_mean(tf.square(y_true - y_pred)))


# ── 3. Model factory ───────────────────────────────────────────────────────
def build_convlstm(input_shape, filters_enc=FILTERS_ENC, filters_dec=FILTERS_DEC,
                   kernel_size=KERNEL_SIZE, dropout=DROPOUT, lr=LR):
    T, H, W, C = input_shape
    ks = (kernel_size, kernel_size)

    inputs = keras.Input(shape=input_shape, name='input_sequence')

    # Encoder
    x = layers.ConvLSTM2D(filters_enc[0], ks, padding='same', return_sequences=True, name='enc_convlstm_1')(inputs)
    x = layers.BatchNormalization(name='enc_bn_1')(x)
    x = layers.ConvLSTM2D(filters_enc[1], ks, padding='same', return_sequences=True, name='enc_convlstm_2')(x)
    x = layers.BatchNormalization(name='enc_bn_2')(x)
    x = layers.ConvLSTM2D(filters_enc[2], ks, padding='same', return_sequences=False, name='enc_convlstm_3')(x)
    x = layers.BatchNormalization(name='enc_bn_3')(x)

    # Bridge
    x = layers.Reshape((1, H, W, filters_enc[2]), name='expand_time')(x)

    # Decoder
    x = layers.ConvLSTM2D(filters_dec[0], ks, padding='same', return_sequences=True, name='dec_convlstm_1')(x)
    x = layers.BatchNormalization(name='dec_bn_1')(x)
    if dropout > 0:
        x = layers.TimeDistributed(layers.SpatialDropout2D(dropout, name='sd1'), name='td_d1')(x)
    x = layers.ConvLSTM2D(filters_dec[1], ks, padding='same', return_sequences=True, name='dec_convlstm_2')(x)
    x = layers.BatchNormalization(name='dec_bn_2')(x)
    if dropout > 0:
        x = layers.TimeDistributed(layers.SpatialDropout2D(dropout, name='sd2'), name='td_d2')(x)

    # Output
    x = layers.TimeDistributed(layers.Conv2D(1, (1,1), activation='sigmoid', padding='same', name='out_conv'), name='td_out')(x)
    outputs = layers.Reshape((H, W, 1), name='squeeze_time')(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name='ConvLSTM_NTL')
    model.compile(optimizer=keras.optimizers.Adam(lr), loss='mae', metrics=[pixel_rmse])
    return model

print('Model factory defined.')

In [ ]:
# ── 4. Load data ───────────────────────────────────────────────────────────
data    = np.load(FRAMES_NPZ)
X_train = data['X_train']
y_train = data['y_train']
X_test  = data['X_test']
y_test  = data['y_test']

print(f'X_train: {X_train.shape}  y_train: {y_train.shape}')
print(f'X_test : {X_test.shape}   y_test : {y_test.shape}')

# Train/val split
n_fit = len(X_train) - VAL_SPLIT
X_fit, y_fit = X_train[:n_fit], y_train[:n_fit]
X_val, y_val = X_train[n_fit:], y_train[n_fit:]
print(f'Train: {n_fit}  Val: {VAL_SPLIT}')

INPUT_SHAPE = tuple(X_fit.shape[1:])   # (12, 79, 79, 1)
print(f'input_shape: {INPUT_SHAPE}')

In [ ]:
# ── 5. Build model ─────────────────────────────────────────────────────────
model = build_convlstm(input_shape=INPUT_SHAPE)
model.summary()
print(f'\nTrainable params: {model.count_params():,}')

In [ ]:
# ── 6. Train ───────────────────────────────────────────────────────────────
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=PATIENCE,
        restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=8, min_lr=1e-6, verbose=1),
    keras.callbacks.ModelCheckpoint(
        filepath=MODEL_OUT, monitor='val_loss',
        save_best_only=True, verbose=0),
]

history = model.fit(
    X_fit, y_fit,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
# ── 7. Save history ────────────────────────────────────────────────────────
hist = {
    'loss':           [float(v) for v in history.history['loss']],
    'val_loss':       [float(v) for v in history.history['val_loss']],
    'pixel_rmse':     [float(v) for v in history.history.get('pixel_rmse', [])],
    'val_pixel_rmse': [float(v) for v in history.history.get('val_pixel_rmse', [])],
    'filters_enc':    list(FILTERS_ENC),
    'filters_dec':    list(FILTERS_DEC),
}
with open(HIST_OUT, 'w') as f:
    json.dump(hist, f, indent=2)
print(f'Saved history  → {HIST_OUT}')
print(f'Saved model    → {MODEL_OUT}')

best_ep   = int(np.argmin(hist['val_loss'])) + 1
best_loss = min(hist['val_loss'])
print(f'Best epoch: {best_ep}  |  Best val_loss (MAE): {best_loss:.4f}')

In [ ]:
# ── 8. Loss curve ──────────────────────────────────────────────────────────
epochs = range(1, len(hist['loss']) + 1)
plt.figure(figsize=(9, 4))
plt.plot(epochs, hist['loss'],     label='Train MAE')
plt.plot(epochs, hist['val_loss'], label='Val MAE')
plt.axvline(best_ep, color='gray', linestyle='--', label=f'Best epoch {best_ep}')
plt.xlabel('Epoch'); plt.ylabel('MAE (scaled [0,1])')
plt.title('ConvLSTM — Training Loss Curve (Kolkata)')
plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'loss_curve.png'), dpi=100)
plt.show()
print('Saved loss_curve.png')

In [ ]:
# ── 9. Package output for download ─────────────────────────────────────────
import zipfile
zip_path = '/kaggle/working/kolkata_convlstm_output.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(MODEL_OUT,  'convlstm_model.keras')
    zf.write(HIST_OUT,   'training_history.json')
    zf.write(os.path.join(OUTPUT_DIR, 'loss_curve.png'), 'loss_curve.png')

size_mb = os.path.getsize(zip_path) / 1e6
print(f'\nDownload ready: kolkata_convlstm_output.zip  ({size_mb:.1f} MB)')
print('Find it under  Output → kolkata_convlstm_output.zip → Download')